In [ ]:
#%pip install -U langchain
#%pip install -U langchain-classic
#%pip install -U langchain-ollama
#%pip install -U langchain-chroma
#%pip install -U tensorflow
#%pip install langchain-chroma -U
#%pip install -U langchain_community
#%pip install pandas -U
#%pip install chromadb -U
#%pip install -U pypdf

In [3]:
import chromadb
from chromadb.utils import embedding_functions
embedding_func = embedding_functions.DefaultEmbeddingFunction()
chroma_client = chromadb.Client()

# Task 1: Create a LangChain vector store with Chroma

In [4]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
embeddings = OllamaEmbeddings(model="mxbai-embed-large")
# create vector store
vector_store = Chroma(collection_name="langchain_chroma_movies", embedding_function=embeddings)


# 2. Use LangChain’s PDF loader and text splitter to process the “Five Movies”

In [5]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
# load file 
filepath = "./dataset/Five Movies.pdf"
loader = PyPDFLoader(filepath)
document=loader.load()

# split 
c_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " "],
    chunk_size=800,
    chunk_overlap=10
)
documents = c_splitter.split_documents(document)



/var/folders/yj/n_z_slv1029_2t6ck3q5hvn80000gn/T/ipykernel_96694/3864857944.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


# 3. Add the processed pdf files to the vector store collection.

In [6]:

# store all the documents in the vector store
ids = [str(i) for i in range(len(documents))]
vector_store.add_documents(documents=documents, ids=ids)

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11']

# 4. Use get() to retrieve all documents in the vector store

In [7]:
# retrieve all documents from the vector store
vector_store.get() 

{'ids': ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11'],
 'embeddings': None,
 'documents': ['12 Angry Men \n12 Angry Men, by Sidney Lumet, may be the most radical courtroom drama in cinema \nhistory. A behind-closed-doors look at the American legal system that is as riveting as it \nis spare, this iconic adaptation of Reginald Rose’s teleplay stars Henry Fonda as the \ndissenting member on a jury of white men ready to pass judgment on a Puerto Rican \nteenager charged with murdering his father. The result is a saga of epic proportions that \nplays out over a tense afternoon in one sweltering room. Lumet’s electrifying snapshot \nof 1950s America on the verge of change is one of the great feature film debuts. \n \nBLU-RAY SPECIAL EDITION FEATURES  \n\uf0b7 New high-definition digital restoration, with uncompressed monaural soundtrack',
  '\uf0b7 Frank Schaffner’s 1955 teleplay of 12 Angry Men, from the series Studio One, \nfeaturing an introduction by Ron Simon, curator 

# 5. Use get() to retrieve all chunks from page 0

In [8]:
print(vector_store.get(where={"page": 0}))

{'ids': ['0', '1', '2'], 'embeddings': None, 'documents': ['12 Angry Men \n12 Angry Men, by Sidney Lumet, may be the most radical courtroom drama in cinema \nhistory. A behind-closed-doors look at the American legal system that is as riveting as it \nis spare, this iconic adaptation of Reginald Rose’s teleplay stars Henry Fonda as the \ndissenting member on a jury of white men ready to pass judgment on a Puerto Rican \nteenager charged with murdering his father. The result is a saga of epic proportions that \nplays out over a tense afternoon in one sweltering room. Lumet’s electrifying snapshot \nof 1950s America on the verge of change is one of the great feature film debuts. \n \nBLU-RAY SPECIAL EDITION FEATURES  \n\uf0b7 New high-definition digital restoration, with uncompressed monaural soundtrack', '\uf0b7 Frank Schaffner’s 1955 teleplay of 12 Angry Men, from the series Studio One, \nfeaturing an introduction by Ron Simon, curator at the Paley Center for Media \n\uf0b7 Production h

# 6. Use get() to retrieve the chunk with id 0

In [9]:
print(vector_store.get(ids=['0']))

{'ids': ['0'], 'embeddings': None, 'documents': ['12 Angry Men \n12 Angry Men, by Sidney Lumet, may be the most radical courtroom drama in cinema \nhistory. A behind-closed-doors look at the American legal system that is as riveting as it \nis spare, this iconic adaptation of Reginald Rose’s teleplay stars Henry Fonda as the \ndissenting member on a jury of white men ready to pass judgment on a Puerto Rican \nteenager charged with murdering his father. The result is a saga of epic proportions that \nplays out over a tense afternoon in one sweltering room. Lumet’s electrifying snapshot \nof 1950s America on the verge of change is one of the great feature film debuts. \n \nBLU-RAY SPECIAL EDITION FEATURES  \n\uf0b7 New high-definition digital restoration, with uncompressed monaural soundtrack'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'producer': 'www.ilovepdf.com', 'total_pages': 5, 'page': 0, 'page_label': '1', 'creationdate': '2026-03-03T14:4

# 7. Update the content of id 0 
by adding the following string to the beginning of the existing
content of id 0. “This document contains information on five different movies in the
Criterion Collection, as well as their Blu Ray re-releases.”

In [10]:
# grab record and store content
existing_record = vector_store.get(ids=['0'])
current_content = existing_record['documents'][0]
current_metadata = existing_record['metadatas'][0]

# prepend the new string to the existing content
new_string = "This document contains information on five different movies in the Criterion Collection, as well as their Blu Ray re-releases. "
updated_content = new_string + current_content

# create a new Document object with new string
updated_doc = Document(page_content=updated_content, id="0", metadata=current_metadata)

# update the vector store using LangChain's update_documents method
vector_store.update_documents(ids=['0'], documents=[updated_doc])

# 8. Query the vector store with similarity_search_with_relevance_scores() and display 4
results. One query you can use is “What are the special features included in the Blu Rays
for these movies?”

In [11]:
cutoff=0.2
results_with_score = vector_store.similarity_search_with_relevance_scores(
    "What are the special features included in the Blu Rays for these movies?",
    k=4,
    score_threshold=cutoff)

for doc, score in results_with_score:
    print(f"Score: {score:.4f}\nContent: {doc.page_content}\nMetadata: {doc.metadata}\n")

Score: 0.5028
Content: This document contains information on five different movies in the Criterion Collection, as well as their Blu Ray re-releases. 12 Angry Men 
12 Angry Men, by Sidney Lumet, may be the most radical courtroom drama in cinema 
history. A behind-closed-doors look at the American legal system that is as riveting as it 
is spare, this iconic adaptation of Reginald Rose’s teleplay stars Henry Fonda as the 
dissenting member on a jury of white men ready to pass judgment on a Puerto Rican 
teenager charged with murdering his father. The result is a saga of epic proportions that 
plays out over a tense afternoon in one sweltering room. Lumet’s electrifying snapshot 
of 1950s America on the verge of change is one of the great feature film debuts. 
 
BLU-RAY SPECIAL EDITION FEATURES  
 New high-definition digital restoration, with uncompressed monaural soundtrack
Metadata: {'producer': 'www.ilovepdf.com', 'page': 0, 'moddate': '2026-03-03T14:46:41+00:00', 'total_pages': 5, '

# 9. Based on the results for the previous task, give a similarity threshold that will remove at
# least one but not all of the results

In [12]:
cutoff=0.4
results_with_score = vector_store.similarity_search_with_relevance_scores(
    "What are the special features included in the Blu Rays for these movies?",
    k=4,
    score_threshold=cutoff)

for doc, score in results_with_score:
    print(f"Score: {score:.4f}\nContent: {doc.page_content}\nMetadata: {doc.metadata}\n")

Score: 0.5028
Content: This document contains information on five different movies in the Criterion Collection, as well as their Blu Ray re-releases. 12 Angry Men 
12 Angry Men, by Sidney Lumet, may be the most radical courtroom drama in cinema 
history. A behind-closed-doors look at the American legal system that is as riveting as it 
is spare, this iconic adaptation of Reginald Rose’s teleplay stars Henry Fonda as the 
dissenting member on a jury of white men ready to pass judgment on a Puerto Rican 
teenager charged with murdering his father. The result is a saga of epic proportions that 
plays out over a tense afternoon in one sweltering room. Lumet’s electrifying snapshot 
of 1950s America on the verge of change is one of the great feature film debuts. 
 
BLU-RAY SPECIAL EDITION FEATURES  
 New high-definition digital restoration, with uncompressed monaural soundtrack
Metadata: {'page_label': '1', 'author': 'Robertson, Michael', 'page': 0, 'total_pages': 5, 'creator': 'Microsoft®

# 10. Create a retriever from the vector store 

In [13]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 10} 
) 

# 11. Define a simple function which takes a list of Document objects as input, 
combines their page_contents into a single string separated by double line breaks, and then
returns that string.

In [14]:
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

# 12. Use Langchain’s ChatPromptTemplate to create a prompt for the chain

In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama.llms import OllamaLLM

model = OllamaLLM(model="llama3.2:3b", temperature=0)

template = """
You are a helpful assistant. 
Your users are asking questions about information contained in the Five Movies file.
If you do not know the answer, please say I do not know.

Here are some relevant information: {information}

Here is the question to answer: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

# 13. Put everything together to create the chain.

In [16]:
#chain = prompt | model
chain = (
    {"information": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

question="What are the special features included in the Blu Rays for these movies?"
response = chain.invoke(question)
print(response)

Here are the special features included in the Blu Rays for the movies:

1. 12 Angry Men:
   - New high-definition digital restoration, with uncompressed monaural soundtrack

2. Seven Samurai:
   - New 4K digital restoration, with the original uncompressed monaural soundtrack and an optional 2.0 surround DTS-HD Master Audio soundtrack
   - Two audio commentaries
   - Making-of documentary, created as part of the Toho Masterworks series Akira Kurosawa: It Is Wonderful to Create
   - My Life in Cinema (1993), a two-hour conversation between director Akira Kurosawa and filmmaker Nagisa Oshima
   - “Seven Samurai”: Origins and Influences, a documentary looking at the samurai traditions and films that helped shape Kurosawa’s masterpiece
   - Trailers and teaser
   - Gallery of rare posters, behind-the-scenes photos, and production stills

3. The Element of Crime:
   - 3K digital restoration, approved by director Lars von Trier, with uncompressed monaural soundtrack
   - Audio commentaries fe

# 14. Think of two more queries for the chain and display their results

In [17]:
# additional question 1
question="Which movies are action movies?"
response = chain.invoke(question)
print(response)

Based on the provided information, the following movies are action movies:

1. Seven Samurai (Shichinin no samurai) - a three-hour epic film with relentless action, philosophy, and entertainment.
2. RoboCop - a science-fiction film with special effects, a grown-up superhero fantasy, and a focus on action and violence.

Note that while 12 Angry Men is a courtroom drama, it also has elements of tension and suspense, which could be considered action-oriented. However, it is not typically classified as an action movie in the same vein as the other two.


In [25]:
# additional question 2
question="What movies have similar themes?"
response = chain.invoke(question)
print(response)

Based on the provided information, it appears that "The Element of Crime" and "Seven Samurai" share similar themes. Both films are known for their:

* Exploration of human emotions and psychology
* Use of philosophical and cultural commentary
* Blend of action, suspense, and drama
* Focus on the human condition and the struggle for survival

Additionally, "The Element of Crime" and "RoboCop" also share some similarities in terms of their:

* Exploration of the human condition and the impact of technology on society
* Use of science fiction and dystopian elements to comment on contemporary issues

However, it's worth noting that each film has its own unique tone and style, and not all of these themes may be present in all three films.


# 15. Use delete to remove the document with id 0 and get to show that it was successful

In [19]:
# Delete operation
vector_store.delete(ids=["0"])
#print(vector_store.get())
# retrieve all the document ids from the vector store
#Get every record in a collection
all_data = vector_store.get()
# Extract and print the document IDs
doc_ids = all_data["ids"]
print(doc_ids)

['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11']
